# POT-PTQ: A Two-step Power-of-Two Post-training Quantization — runnable demo

Self-contained reproduction of **"POT-PTQ: A Two-step Power-of-Two
Post-training for LLMs"** (Wang et al., arXiv:2507.11959), applied to a
small char-level GPT (a few million parameters) instead of LLaMA so the
*entire* pipeline — training, Step 1 (data-agnostic scale init),
Step 2 (data-dependent fine-tuning), evaluation, and the Section 4
bitwise dequantization kernel — runs top-to-bottom in one sitting on
CPU **or** CUDA.

**Just run every cell in order** (`Runtime -> Run all` in Colab). The
code auto-detects a GPU (`torch.cuda.is_available()`) and uses it if
present; on CPU-only machines the default sizes below are kept small so
it still finishes in a few minutes. Bump up `STEPS`, `D_MODEL`,
`N_LAYER`, `CALIB_SEQS`, and the epoch counts in the config cell once
you've confirmed it runs, to get results closer to the paper's actual
training/calibration budget.

Every equation/algorithm from the paper is implemented and labelled in
comments (Eq. 1-18, Algorithm 1, Algorithm 2, Fig. 4). See the markdown
notes before each section for the paper mapping.


## 0. Setup

In [ ]:
# Installs torch if it isn't already available. On Colab this cell is a
# no-op (torch is preinstalled); on a fresh machine it pulls the CUDA
# build automatically if a GPU + matching CUDA driver is present.
import importlib.util, subprocess, sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


In [ ]:
# Tiny-Shakespeare corpus (public domain, hosted on GitHub) -- used both
# to train the tiny baseline GPT and as the small "calibration set" for
# Step 2, playing the role WikiText-2 plays in the paper (Sec. 5.1).
import os, urllib.request

DATA_PATH = "input.txt"
if not os.path.exists(DATA_PATH):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, DATA_PATH)
print(f"{os.path.getsize(DATA_PATH)/1e6:.2f} MB of text downloaded to {DATA_PATH}")


## 1. Config

Small-but-real defaults so the whole notebook finishes quickly on CPU.
**On a GPU, increase these** (commented suggestions given) to get a
bigger model / longer calibration, closer to the paper's own budget
(128 calibration sequences of 2048 tokens, 10 epochs @ 3-bit / 40 epochs
@ 2-bit, Sec. 5.1).


In [ ]:
# ---- model size (a "multi-million parameter" GPT, like the paper's
# LLaMA models but scaled down) ----
BLOCK_SIZE = 128        # GPU suggestion: 256
D_MODEL    = 192        # GPU suggestion: 384-512
N_LAYER    = 4           # GPU suggestion: 8-12
N_HEAD     = 4           # GPU suggestion: 8

# ---- baseline training ----
TRAIN_STEPS = 400        # GPU suggestion: 3000-5000
TRAIN_BATCH = 32         # GPU suggestion: 64
LR          = 3e-4

# ---- POT-PTQ quantization (Sec. 5.1) ----
GROUP_SIZE     = 64       # paper uses 64 or 128
CALIB_SEQS     = 32       # paper: 128
CALIB_SEQ_LEN  = 128      # paper: 2048
EPOCHS_3BIT    = 4        # paper: 10
EPOCHS_2BIT    = 6        # paper: 40
QUANT_LR       = 1e-3     # matches paper Sec. 5.1
WEIGHT_DECAY   = 1e-1     # matches paper Sec. 5.1
EVAL_BATCHES   = 20

print("Config set.")


## 2. Model (`TinyGPT`)

A standard pre-LN decoder-only transformer. It has exactly the weight
matrices the paper quantizes: `Q/K/V/O` projections in attention and
`fc1/fc2` in the MLP (Eq. 11-13). Embeddings, LayerNorms, and the output
head are kept full precision, matching the weight-only PTQ setting the
paper (and GPTQ/AWQ/OmniQuant, which it compares against) uses.


In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head, block_size, dropout=0.0):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_model = d_model
        self.head_dim = d_model // n_head

        # Separate projections so each is quantized as its own weight
        # matrix W^(l), matching Eq. (12): Q,K,V,O.
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.o_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, d_model, mlp_ratio=4, dropout=0.0):
        super().__init__()
        # W1, W2 -> matches Eq. (13): MLP(X) = sigma(X W1) W2
        self.fc1 = nn.Linear(d_model, mlp_ratio * d_model, bias=False)
        self.fc2 = nn.Linear(mlp_ratio * d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = self.fc2(x)
        return self.dropout(x)


class Block(nn.Module):
    """One transformer block: X + Attn(X) then MLP, matches Eq. (11)."""

    def __init__(self, d_model, n_head, block_size, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    """A char-level GPT with a few million parameters."""

    def __init__(self, vocab_size, block_size=128, d_model=256, n_layer=6,
                 n_head=8, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, n_head, block_size, mlp_ratio, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, "sequence longer than block_size"
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("TinyGPT defined.")


## 3. Data: char-level tokenizer + batching

In [ ]:
class CharDataset:
    """Minimal char-level tokenizer + train/val/calib splitter."""

    def __init__(self, path, block_size=128, val_fraction=0.1):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        data = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)

        n = int(len(data) * (1 - val_fraction))
        self.train_data = data[:n]
        self.val_data = data[n:]
        self.block_size = block_size

    def encode(self, s):
        return torch.tensor([self.stoi[c] for c in s], dtype=torch.long)

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

    def get_batch(self, split, batch_size, device):
        data = self.train_data if split == "train" else self.val_data
        ix = torch.randint(len(data) - self.block_size - 1, (batch_size,))
        x = torch.stack([data[i:i + self.block_size] for i in ix])
        y = torch.stack([data[i + 1:i + self.block_size + 1] for i in ix])
        return x.to(device), y.to(device)

    def get_calibration_batches(self, n_seqs, seq_len, device, seed=0):
        """Sample n_seqs random contiguous sequences from the TRAIN split,
        the analogue of the paper's 128 random 2048-token WikiText-2
        sequences used for Step 2 (Sec. 5.1)."""
        g = torch.Generator().manual_seed(seed)
        data = self.train_data
        ix = torch.randint(len(data) - seq_len - 1, (n_seqs,), generator=g)
        xs = torch.stack([data[i:i + seq_len] for i in ix]).to(device)
        return xs

ds = CharDataset(DATA_PATH, block_size=BLOCK_SIZE)
print(f"vocab_size={ds.vocab_size}  train_chars={len(ds.train_data)}  val_chars={len(ds.val_data)}")


## 4. Train the FP32 baseline

In [ ]:
model = TinyGPT(ds.vocab_size, block_size=BLOCK_SIZE, d_model=D_MODEL,
                 n_layer=N_LAYER, n_head=N_HEAD).to(DEVICE)
print(f"model params: {model.num_params()/1e6:.3f}M")

opt = torch.optim.AdamW(model.parameters(), lr=LR)
for step in range(TRAIN_STEPS):
    x, y = ds.get_batch("train", TRAIN_BATCH, DEVICE)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % max(1, TRAIN_STEPS // 10) == 0 or step == TRAIN_STEPS - 1:
        model.eval()
        with torch.no_grad():
            xv, yv = ds.get_batch("val", TRAIN_BATCH, DEVICE)
            _, vloss = model(xv, yv)
        model.train()
        print(f"step {step:5d} | train loss {loss.item():.4f} | val loss {vloss.item():.4f}")

model.eval()
print("Baseline training done.")


In [ ]:
# quick sanity-check generation from the FP32 baseline
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
with torch.no_grad():
    out = model.generate(context, max_new_tokens=200, temperature=0.9, top_k=20)
print(ds.decode(out[0].tolist()))


## 5. POT-PTQ core (Sec. 3, Eq. 1-18, Algorithm 1)

Weight representation (Eq. 1 / Eq. 4): `W_tilde = S * P * 2^E`
- `S`: positive per-group scale (learnable in Step 2)
- `P`: sign, `P = sign(W)` (frozen after Step 1)
- `E`: exponent in `[0, qmax]`, `qmax = 2^(n-1) - 1` for `n`-bit PoT (Eq. 2)

Group-wise quantization (Eq. 5): weights are grouped along the input
(reduction) dimension of each `nn.Linear` weight in contiguous chunks of
`group_size` (the paper: "each column of `W(l)` is divided into
fixed-size groups (e.g., 64 or 128 rows)").


In [ ]:
# ---------------- Straight-Through Estimator (Eq. 18) ----------------
def ste_round_clamp(x: torch.Tensor, qmax: int) -> torch.Tensor:
    """round(clamp(x)) in forward, identity gradient in backward
    (Straight-Through Estimator, used for Eq. 16/18)."""
    rounded = torch.clamp(torch.round(x), 0, qmax)
    return x + (rounded - x).detach()


def qmax_for_bits(n_bits: int) -> int:
    """qmax = 2^(n-1) - 1, Eq. (2)."""
    return 2 ** (n_bits - 1) - 1


# ---------------- group reshaping helpers (Eq. 5) ----------------
def to_groups(W: torch.Tensor, group_size: int):
    out_f, in_f = W.shape
    pad = (-in_f) % group_size
    if pad:
        W = F.pad(W, (0, pad))
    Wg = W.view(out_f, -1, group_size)
    return Wg, pad


def from_groups(Wg: torch.Tensor, pad: int, in_f: int):
    W = Wg.reshape(Wg.shape[0], -1)
    if pad:
        W = W[:, :in_f]
    return W


# ---------------- Algorithm 1: Parallel Data-Agnostic Scale Init ----------------
@torch.no_grad()
def data_agnostic_scale_init(W: torch.Tensor, n_bits: int, group_size: int = 64,
                              n_candidates: int = 200, device=None):
    """Vectorized Algorithm 1: for every group, grid-search multipliers
    B = {0.01*i | i=1..n_candidates} of the base scale
    s0 = max|W_group| / (2^qmax - 1)                          (Eq. 10)
    to minimize the weight reconstruction MSE (Eq. 8/9), fully vectorized
    across all groups & candidates (the paper's "parallelized across GPU
    threads")."""
    device = device or W.device
    qmax = qmax_for_bits(n_bits)
    Wg, pad = to_groups(W.to(device), group_size)          # (O, G, g)
    O, G, g = Wg.shape
    absW = Wg.abs()
    P = torch.sign(Wg)
    P[P == 0] = 1.0

    s0 = absW.amax(dim=-1, keepdim=True) / max(qmax, 1)    # (O, G, 1), Eq.10
    s0 = torch.clamp(s0, min=1e-12)

    B = torch.arange(1, n_candidates + 1, device=device, dtype=W.dtype) * 0.01
    sb = s0.unsqueeze(-1) * B.view(1, 1, 1, -1)             # (O, G, 1, C)
    sb = torch.clamp(sb, min=1e-12)

    absW_ = absW.unsqueeze(-1)                              # (O, G, g, 1)
    E = torch.clamp(torch.round(torch.log2(absW_ / sb)), 0, qmax)  # Eq.6
    Wg_hat = sb * P.unsqueeze(-1) * torch.pow(2.0, E)        # Eq.7

    Q1 = ((Wg.unsqueeze(-1) - Wg_hat) ** 2).sum(dim=2)       # Eq.8
    best_idx = torch.argmin(Q1, dim=-1)

    b_star = B[best_idx]
    S = (s0.squeeze(-1) * b_star).unsqueeze(-1)               # Eq.9

    return S, P, pad


# ---------------- PoTLinear: drop-in nn.Linear replacement ----------------
class PoTLinear(nn.Module):
    """Quantized linear layer. Holds frozen sign P (Eq.1), Step-1 scale S
    (Eq.9), and learnable residual Gamma (Eq.15) that Step 2 fine-tunes.
    Dequantized weight W_tilde = Shat * P * 2^E(Shat) recomputed on the
    fly (Eq. 16-17)."""

    def __init__(self, linear: nn.Linear, n_bits: int, group_size: int = 64):
        super().__init__()
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.n_bits = n_bits
        self.group_size = group_size
        self.qmax = qmax_for_bits(n_bits)

        W = linear.weight.data.clone()
        S, P, pad = data_agnostic_scale_init(W, n_bits, group_size)
        self.pad = pad
        self.register_buffer("W_orig_grouped", to_groups(W, group_size)[0])
        self.register_buffer("P", P)
        self.register_buffer("S", S)                 # Step-1 scale, frozen
        self.Gamma = nn.Parameter(torch.zeros_like(S))  # Eq.15, Step-2 learnable

        self.bias = None
        if linear.bias is not None:
            self.bias = nn.Parameter(linear.bias.data.clone())

    def dequantized_weight(self):
        Shat = self.S * (1.0 + self.Gamma)                        # Eq.15
        Shat_c = torch.clamp(Shat.abs(), min=1e-12) * torch.sign(Shat + 1e-12)
        absW = self.W_orig_grouped.abs()
        E = ste_round_clamp(torch.log2(absW / Shat_c.abs()), self.qmax)   # Eq.16 (+STE)
        Wg_hat = Shat_c * self.P * torch.pow(2.0, E)               # Eq.17
        W_hat = from_groups(Wg_hat, self.pad, self.in_features)
        return W_hat

    def forward(self, x):
        W_hat = self.dequantized_weight()
        return F.linear(x, W_hat, self.bias)

    def freeze(self):
        """Bake Gamma into S, reset Gamma to 0 (before quantizing the next
        block using this block's *quantized* output as input)."""
        with torch.no_grad():
            self.S.mul_(1.0 + self.Gamma)
            self.Gamma.zero_()

    def weight_only_mse(self):
        with torch.no_grad():
            W_hat = self.dequantized_weight()
            Wg_hat, _ = to_groups(W_hat, self.group_size)
            return ((Wg_hat - self.W_orig_grouped) ** 2).mean().item()

print("PoT quantization core defined.")


## 6. Step 2 driver: block-wise sequential fine-tuning (Algorithm 2, Eq. 11-14)

For each transformer block, in order:
1. **Step 1** (already run inside `PoTLinear.__init__`): data-agnostic
   grid-search init for every `nn.Linear` in the block.
2. **Step 2**: push calibration activations through the block with the
   *original* weights (`Horig`) and with the *quantized* weights
   (`Hquant`), and fine-tune only the `Gamma` parameters of that block's
   `PoTLinear` layers with Adam to minimize `‖Horig - Hquant‖² + (λ/2)‖Γ‖²`
   (Eq. 14), using the STE gradient (Eq. 18) to backprop through the
   rounding operation.
3. Freeze `Gamma` into `S`, and propagate this block's **quantized**
   output forward as the input to the next block's calibration — the
   same sequential calibration strategy used by GPTQ/AWQ/OmniQuant, and
   a direct reading of the paper's per-block objective `F^(l)`.


In [ ]:
import copy, math


def _get_linears(block):
    return {
        "attn.q_proj": block.attn.q_proj, "attn.k_proj": block.attn.k_proj,
        "attn.v_proj": block.attn.v_proj, "attn.o_proj": block.attn.o_proj,
        "mlp.fc1": block.mlp.fc1, "mlp.fc2": block.mlp.fc2,
    }


def _set_linear(block, name, module):
    if name == "attn.q_proj": block.attn.q_proj = module
    elif name == "attn.k_proj": block.attn.k_proj = module
    elif name == "attn.v_proj": block.attn.v_proj = module
    elif name == "attn.o_proj": block.attn.o_proj = module
    elif name == "mlp.fc1": block.mlp.fc1 = module
    elif name == "mlp.fc2": block.mlp.fc2 = module
    else: raise ValueError(name)


@torch.no_grad()
def embed(m, idx):
    B, T = idx.shape
    pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
    return m.drop(m.tok_emb(idx) + m.pos_emb(pos))


def quantize_model(model, calib_x, n_bits, group_size=64, epochs=10,
                    lr=1e-3, weight_decay=1e-1, device="cpu", verbose=True):
    """Quantize `model` in-place with POT-PTQ at `n_bits` precision."""
    model = model.to(device)
    calib_x = calib_x.to(device)

    ref_model = copy.deepcopy(model).to(device)   # frozen FP reference for Horig
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    with torch.no_grad():
        X = embed(ref_model, calib_x)
    X_quant = X.clone()

    stats = []
    for li, (blk, ref_blk) in enumerate(zip(model.blocks, ref_model.blocks)):
        linears = _get_linears(blk)
        pot_layers = {}
        for name, lin in linears.items():
            pot = PoTLinear(lin, n_bits=n_bits, group_size=group_size).to(device)
            pot_layers[name] = pot
            _set_linear(blk, name, pot)

        params = [pot.Gamma for pot in pot_layers.values()]
        for p in params:
            p.requires_grad_(True)
        opt = torch.optim.Adam(params, lr=lr)

        with torch.no_grad():
            Horig = ref_blk(X_quant)              # Eq.14: F^(l)(W^(l), X)

        n = X_quant.shape[0]
        batch_size = min(16, n)
        loss_val = None
        for epoch in range(epochs):
            perm = torch.randperm(n, device=device)
            for s in range(0, n, batch_size):
                bidx = perm[s:s + batch_size]
                xb, hb = X_quant[bidx], Horig[bidx]
                Hquant = blk(xb)
                rec_loss = torch.mean((hb - Hquant) ** 2)
                reg = sum((p ** 2).sum() for p in params)
                loss = rec_loss + (weight_decay / 2.0) * reg
                opt.zero_grad(); loss.backward(); opt.step()
                loss_val = rec_loss.item()

        for pot in pot_layers.values():
            pot.freeze()
            for p in pot.parameters():
                p.requires_grad_(False)

        with torch.no_grad():
            X_quant = blk(X_quant)

        w_mse = sum(pot.weight_only_mse() for pot in pot_layers.values()) / len(pot_layers)
        stats.append({"block": li, "recon_loss": loss_val, "weight_mse": w_mse})
        if verbose:
            print(f"  [block {li:2d}] output-recon MSE={loss_val:.6e}  avg weight MSE={w_mse:.6e}")

    return model, stats


@torch.no_grad()
def evaluate_perplexity(model, dataset, device, split="val", n_batches=20, batch_size=16):
    model.eval()
    total_loss, total_count = 0.0, 0
    for _ in range(n_batches):
        x, y = dataset.get_batch(split, batch_size, device)
        _, loss = model(x, y)
        total_loss += loss.item() * x.numel()
        total_count += x.numel()
    avg_loss = total_loss / total_count
    return math.exp(avg_loss), avg_loss

print("Step-2 driver defined.")


## 7. Run POT-PTQ at 3-bit and 2-bit (Table 1 style)

This is the main event: quantize a fresh copy of the trained baseline at
3-bit and 2-bit precision and compare perplexity against the FP32
baseline, the same comparison the paper makes in Table 1.


In [ ]:
base_ppl, base_loss = evaluate_perplexity(model, ds, DEVICE, n_batches=EVAL_BATCHES)
print(f"[FP32 baseline]   val loss={base_loss:.4f}  perplexity={base_ppl:.3f}")

calib_x = ds.get_calibration_batches(CALIB_SEQS, CALIB_SEQ_LEN, DEVICE)

results = {"fp32": base_ppl}
quantized_models = {}

for n_bits, epochs in [(3, EPOCHS_3BIT), (2, EPOCHS_2BIT)]:
    print(f"\n===== POT-PTQ {n_bits}-bit (Step1 + Step2, {epochs} epochs) =====")
    qmodel = copy.deepcopy(model)
    qmodel, stats = quantize_model(qmodel, calib_x, n_bits=n_bits, group_size=GROUP_SIZE,
                                    epochs=epochs, lr=QUANT_LR, weight_decay=WEIGHT_DECAY,
                                    device=DEVICE, verbose=True)
    ppl, loss = evaluate_perplexity(qmodel, ds, DEVICE, n_batches=EVAL_BATCHES)
    print(f"[POT-PTQ {n_bits}-bit]  val loss={loss:.4f}  perplexity={ppl:.3f}")
    results[f"pot_{n_bits}bit"] = ppl
    quantized_models[n_bits] = qmodel

print("\n===== Summary (Table 1 style) =====")
for k, v in results.items():
    print(f"{k:12s}: perplexity = {v:.3f}")


## 8. Ablation: Step 1 only vs. Step 2 only vs. both (Table 4 style)

Confirms the paper's central claim: Step 1 (robust init) and Step 2
(output-aligned fine-tuning) are complementary — each alone
underperforms using both together.


In [ ]:
print("===== Ablation @ 2-bit =====")

# Step 1 only: zero fine-tuning epochs -> Gamma stays 0, S = pure Step-1 grid-search scale.
qm1 = copy.deepcopy(model)
qm1, _ = quantize_model(qm1, calib_x, n_bits=2, group_size=GROUP_SIZE, epochs=0,
                         lr=QUANT_LR, weight_decay=WEIGHT_DECAY, device=DEVICE, verbose=False)
ppl1, _ = evaluate_perplexity(qm1, ds, DEVICE, n_batches=EVAL_BATCHES)
print(f"Step1 only                  : perplexity={ppl1:.3f}")

# Step 2 only: replace Algorithm-1's grid search with the naive b=1 init (s=s0),
# then fine-tune Gamma as usual -- isolates the value of the grid search itself.
_orig_init = data_agnostic_scale_init

def _naive_init(W, n_bits, group_size=64, n_candidates=200, device=None):
    return _orig_init(W, n_bits, group_size=group_size, n_candidates=1, device=device)

globals()['data_agnostic_scale_init'] = _naive_init
qm2 = copy.deepcopy(model)
qm2, _ = quantize_model(qm2, calib_x, n_bits=2, group_size=GROUP_SIZE, epochs=EPOCHS_2BIT,
                         lr=QUANT_LR, weight_decay=WEIGHT_DECAY, device=DEVICE, verbose=False)
globals()['data_agnostic_scale_init'] = _orig_init
ppl2, _ = evaluate_perplexity(qm2, ds, DEVICE, n_batches=EVAL_BATCHES)
print(f"Step2 only (naive b=1 init) : perplexity={ppl2:.3f}")

print(f"Step1 + Step2 (full method) : perplexity={results['pot_2bit']:.3f}")


## 9. Section 4: hardware-friendly bitwise dequantization (Fig. 4)

Standard dequantization needs a float multiply: `w = scale * q` (Eq. 19).
PoT dequantization instead reconstructs `w = (-1)^sign * 2^E * scale`
(Eq. 20-21). Multiplying an FP16 number by a power of two is *exactly*
the same as adding `E` to the 5-bit exponent field of its raw bit
pattern (and XOR-ing the sign bit) — this cell implements that trick
with pure integer bit ops (no float multiply) and checks it's bit-exact
against the naive formula.


In [ ]:
import time

def dequant_naive_fp(scale, sign, exponent):
    """Standard floating-point dequantization: w = sign * scale * 2^E."""
    return sign * scale.to(torch.float16) * torch.pow(2.0, exponent.to(torch.float16))


def dequant_bitwise(scale, sign, exponent):
    """PoT-PTQ style dequantization: bit manipulation + integer addition
    only (Fig. 4, Sec. 4.2), no floating-point multiply."""
    assert scale.dtype == torch.float16
    scale = scale.contiguous()
    scale_bits = scale.view(torch.int16).to(torch.int32) & 0xFFFF

    MANTISSA_BITS = 10
    exp_shift = (exponent.to(torch.int32) & 0x1F) << MANTISSA_BITS

    orig_sign_bit = (scale_bits >> 15) & 0x1
    magnitude_bits = scale_bits & 0x7FFF
    new_magnitude = (magnitude_bits + exp_shift) & 0x7FFF   # integer addition (Sec 4.2 (b))

    pot_sign_bit = torch.where(sign < 0, torch.ones_like(orig_sign_bit), torch.zeros_like(orig_sign_bit))
    final_sign_bit = orig_sign_bit ^ pot_sign_bit            # bit manipulation (Sec 4.2 (a))

    result_bits = (new_magnitude | (final_sign_bit << 15)).to(torch.int16)
    return result_bits.view(torch.float16)


torch.manual_seed(0)
n = 500_000
scale = (torch.rand(n, device=DEVICE) * 0.05 + 1e-3).to(torch.float16)
sign = torch.where(torch.rand(n, device=DEVICE) > 0.5, 1.0, -1.0).to(torch.float16)
exponent = torch.randint(0, 4, (n,), device=DEVICE)  # 3-bit PoT -> qmax=3

w_naive = dequant_naive_fp(scale, sign, exponent)
w_bitwise = dequant_bitwise(scale, sign, exponent)
max_err = (w_naive.float() - w_bitwise.float()).abs().max().item()
print(f"Correctness: naive-vs-bitwise max abs error = {max_err:.6g} "
      f"(match={torch.allclose(w_naive.float(), w_bitwise.float())})")

def timeit(fn, iters=20):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn(scale, sign, exponent)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters

t_naive, t_bitwise = timeit(dequant_naive_fp), timeit(dequant_bitwise)
print(f"float-multiply dequant : {t_naive*1e3:.4f} ms")
print(f"bit-manipulation dequant: {t_bitwise*1e3:.4f} ms")
if DEVICE == "cpu":
    print("\nNote: this environment has no CUDA GPU. The paper's reported "
          "3.67x/1.63x speedups (Table 6) come from a fused CUDA kernel; "
          "on CPU integer ops don't have the same throughput edge over "
          "float multiply. The formula above is correct and GPU-ready "
          "(runs unmodified if DEVICE == 'cuda').")


## 10. (Optional) Generate text from the quantized model

Sanity check that the 3-bit/2-bit quantized models still produce
sensible output.


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
for n_bits, qm in quantized_models.items():
    qm.eval()
    with torch.no_grad():
        out = qm.generate(context, max_new_tokens=200, temperature=0.9, top_k=20)
    print(f"--- {n_bits}-bit POT-PTQ sample ---")
    print(ds.decode(out[0].tolist()))
    print()
